# What diseases are in MIMIC-IV? An ICD analysis

This notebook answers, from the data rather than from memory:

1. **What does an ICD code actually tell you?**
2. **What is the hierarchy** — chapter, block, category, code — and what does each level add?
3. **How many broad chapters** appear in MIMIC, and **how many unique codes sit in each**?
4. **Do patients have one disease or many?** With real worked examples of both.
5. **How far does one patient's diagnosis set spread** across the hierarchy?
6. **Which conditions travel together?**

It reads only two tables from the MIMIC-IV `hosp` module — `diagnoses_icd` and
`d_icd_diagnoses` — and optionally `admissions` for patient counts. Everything else is derived.

> **Attach the data first.** In the Kaggle editor use **+ Add Input** and attach a MIMIC-IV
> dataset. The next cell finds the files wherever they landed, so no path editing is needed.
> The MIMIC-IV demo (~100 patients) works and runs in seconds; the full credentialed dataset
> gives the real numbers. The notebook reports which one it is reading.

## 0. Locate and load the data

In [ ]:
import os, re, glob, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

SEARCH_ROOTS = ["/kaggle/input", "../input", "./data", "."]

def find_table(*names):
    """
    Locate a MIMIC table by name, whatever the dataset's folder layout is.

    Kaggle mirrors of MIMIC differ in nesting and compression, so every root is
    searched recursively for csv / csv.gz / parquet, and the largest match wins
    -- a 'sample_' file usually sits next to the real one and is far smaller.
    """
    hits = []
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for name in names:
            for ext in ("csv", "csv.gz", "parquet"):
                hits += glob.glob(f"{root}/**/{name}.{ext}", recursive=True)
    if not hits:
        return None
    return max(set(hits), key=os.path.getsize)

def read_table(path, **kw):
    if path.endswith(".parquet"):
        return pd.read_parquet(path, **({"columns": kw["usecols"]} if "usecols" in kw else {}))
    return pd.read_csv(path, **kw)

dx_path  = find_table("diagnoses_icd", "sample_diagnoses_icd")
dic_path = find_table("d_icd_diagnoses", "sample_d_icd_diagnoses")
adm_path = find_table("admissions", "sample_admissions")

print("diagnoses_icd   :", dx_path or "NOT FOUND")
print("d_icd_diagnoses :", dic_path or "NOT FOUND")
print("admissions      :", adm_path or "not found (optional)")

if not dx_path or not dic_path:
    raise SystemExit(
        "Attach a MIMIC-IV dataset with + Add Input. Needs diagnoses_icd and d_icd_diagnoses."
    )

In [ ]:
dx = read_table(dx_path, usecols=["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"])
dic = read_table(dic_path, usecols=["icd_code", "icd_version", "long_title"])

# The two tables disagree on padding and case in some releases, and the same
# string is a valid code in BOTH ICD versions meaning different things -- so the
# join key is always the (code, version) PAIR, never the code alone.
for frame in (dx, dic):
    frame["icd_code"] = frame["icd_code"].astype(str).str.strip().str.upper()
    frame["icd_version"] = pd.to_numeric(frame["icd_version"], errors="coerce").astype("Int64")

dic = dic.drop_duplicates(subset=["icd_code", "icd_version"])
before = len(dx)
dx = dx.merge(dic, on=["icd_code", "icd_version"], how="left")
assert len(dx) == before, "the dictionary join duplicated rows"

unmapped = dx.long_title.isna().mean()
dx["long_title"] = dx.long_title.fillna("Unmapped ICD code " + dx.icd_code)

scale = "FULL MIMIC-IV" if len(dx) > 500_000 else ("DEMO subset" if len(dx) > 500 else "SAMPLE file")
print(f"reading: {scale}")
print(f"  diagnosis rows : {len(dx):,}")
print(f"  admissions     : {dx.hadm_id.nunique():,}")
print(f"  patients       : {dx.subject_id.nunique():,}")
print(f"  unmapped codes : {unmapped:.2%}")
dx.head(8)

## 1. What does an ICD code actually tell you?

A row of `diagnoses_icd` carries four things, and each answers a different question:

| Field | Question it answers |
|---|---|
| `icd_code` | **What** condition — the diagnosis itself |
| `icd_version` | **Which dictionary** the code belongs to, 9 or 10. The same string means different things in each |
| `seq_num` | **How central** it was to this stay. `1` is the *principal* diagnosis, the condition chiefly responsible for the admission |
| `hadm_id` | **Which stay** it was coded on — codes belong to an admission, not to a person |

Two things an ICD code does **not** tell you:

- **Severity.** Two patients with the same code can be worlds apart clinically.
- **When it started.** A code on this admission may be a chronic condition of twenty years'
  standing or something that developed yesterday. Discharge coding does not distinguish them.

The code string itself is structured. Reading `I50.23`:

```
I        chapter letter        diseases of the circulatory system
I50      three-char category   heart failure
I50.2    subcategory           systolic (congestive) heart failure
I50.23   full code             acute on chronic systolic heart failure
```

Each character to the right narrows the meaning. ICD-10 uses those trailing characters for
laterality, episode of care and severity, which is why one clinical concept can occupy dozens of
codes.

In [ ]:
# One admission, exactly as MIMIC stores it. seq_num 1 is the principal diagnosis.
sizes = dx.groupby("hadm_id").size()
demo_hadm = sizes[(sizes >= 8) & (sizes <= 14)].index[0] if ((sizes >= 8) & (sizes <= 14)).any() \
            else sizes.idxmax()

one = dx[dx.hadm_id == demo_hadm].sort_values("seq_num")
print(f"admission {demo_hadm} -- {len(one)} diagnoses coded\n")
for _, r in one.iterrows():
    role = "PRINCIPAL" if r.seq_num == 1 else f"seq {int(r.seq_num)}"
    print(f"  {role:<10} {r.icd_code:<9} ICD-{r.icd_version}  {r.long_title[:88]}")

## 2. The hierarchy

ICD is a tree. Four levels matter here:

```
Chapter      Diseases of the circulatory system      I00-I99     ~20 of these exist
  Block      Heart failure and other heart disease   I30-I52     a published grouping
    Category Heart failure                           I50         the 3-character stem
      Code   Acute on chronic systolic HF            I50.23      what gets stored
```

The mapping below is applied to every code in the data. ICD-9 and ICD-10 divide the body
differently and name the same chapter differently, so equivalent chapters are merged onto one
canonical label — otherwise every chapter appears twice purely because of the coding era.

In [ ]:
ICD10_CHAPTERS = [
    ("A00","B99","Infectious and parasitic diseases"),
    ("C00","D49","Neoplasms"),
    ("D50","D89","Blood and immune mechanism"),
    ("E00","E89","Endocrine, nutritional and metabolic"),
    ("F01","F99","Mental, behavioural and neurodevelopmental"),
    ("G00","G99","Nervous system and sense organs"),
    ("H00","H59","Nervous system and sense organs"),
    ("H60","H95","Nervous system and sense organs"),
    ("I00","I99","Circulatory system"),
    ("J00","J99","Respiratory system"),
    ("K00","K95","Digestive system"),
    ("L00","L99","Skin and subcutaneous tissue"),
    ("M00","M99","Musculoskeletal system and connective tissue"),
    ("N00","N99","Genitourinary system"),
    ("O00","O9A","Pregnancy, childbirth and the puerperium"),
    ("P00","P96","Perinatal conditions"),
    ("Q00","Q99","Congenital malformations"),
    ("R00","R99","Symptoms, signs and abnormal findings"),
    ("S00","T88","Injury and poisoning"),
    ("U00","U85","Codes for special purposes, incl. COVID-19"),
    ("V00","Y99","External causes of morbidity"),
    ("Z00","Z99","Factors influencing health status"),
]

ICD9_CHAPTERS = [
    (1,139,"Infectious and parasitic diseases"),
    (140,239,"Neoplasms"),
    (240,279,"Endocrine, nutritional and metabolic"),
    (280,289,"Blood and immune mechanism"),
    (290,319,"Mental, behavioural and neurodevelopmental"),
    (320,389,"Nervous system and sense organs"),
    (390,459,"Circulatory system"),
    (460,519,"Respiratory system"),
    (520,579,"Digestive system"),
    (580,629,"Genitourinary system"),
    (630,679,"Pregnancy, childbirth and the puerperium"),
    (680,709,"Skin and subcutaneous tissue"),
    (710,739,"Musculoskeletal system and connective tissue"),
    (740,759,"Congenital malformations"),
    (760,779,"Perinatal conditions"),
    (780,799,"Symptoms, signs and abnormal findings"),
    (800,999,"Injury and poisoning"),
]

def chapter_of(code, version):
    """
    Chapter for one (code, version) pair.

    Version is taken from the data rather than guessed from the string, which
    matters: 'E6601' is obesity in ICD-10 and an external-cause code in ICD-9,
    and only the version field separates them.
    """
    c = str(code).strip().upper()
    if not c:
        return "Not coded"
    if version == 9:
        if c[0] == "E":
            return "External causes of morbidity"
        if c[0] == "V":
            return "Factors influencing health status"
        try:
            head = int(c[:3])
        except ValueError:
            return "Unclassified"
        for lo, hi, name in ICD9_CHAPTERS:
            if lo <= head <= hi:
                return name
        return "Unclassified"
    head = c[:3]
    for lo, hi, name in ICD10_CHAPTERS:
        if lo <= head <= hi:
            return name
    return "Unclassified"

dx["chapter"] = [chapter_of(c, v) for c, v in zip(dx.icd_code, dx.icd_version)]
dx["category"] = dx.icd_code.str.slice(0, 3)          # the 3-character stem
print("codes that could not be placed in a chapter:",
      f"{(dx.chapter == 'Unclassified').mean():.3%}")
dx[["icd_code", "icd_version", "category", "chapter", "long_title"]].head(6)

In [ ]:
# The hierarchy, drilled through on real data. Blocks are a published grouping
# between chapter and category; the circulatory ones are listed here as the
# worked example rather than reproducing all ~280 from memory.
CIRC_BLOCKS = [
    ("I00","I02","Acute rheumatic fever"),
    ("I05","I09","Chronic rheumatic heart disease"),
    ("I10","I16","Hypertensive diseases"),
    ("I20","I25","Ischaemic heart diseases"),
    ("I26","I28","Pulmonary heart disease"),
    ("I30","I52","Other forms of heart disease"),
    ("I60","I69","Cerebrovascular diseases"),
    ("I70","I79","Arteries, arterioles and capillaries"),
    ("I80","I89","Veins, lymphatics and lymph nodes"),
    ("I95","I99","Other and unspecified circulatory disorders"),
]

def block_of(code):
    head = str(code)[:3].upper()
    for lo, hi, name in CIRC_BLOCKS:
        if lo <= head <= hi:
            return f"{lo}-{hi}  {name}"
    return None

circ10 = dx[(dx.chapter == "Circulatory system") & (dx.icd_version == 10)].copy()
circ10["block"] = circ10.icd_code.map(block_of)

print("CHAPTER  Circulatory system  (ICD-10 rows only)")
print(f"   {len(circ10):,} diagnosis rows, {circ10.icd_code.nunique():,} distinct codes, "
      f"{circ10.category.nunique():,} categories\n")
print("  BLOCKS")
for blk, g in circ10.groupby("block"):
    print(f"    {blk:<52} {g.icd_code.nunique():>4} codes  {len(g):>7,} rows")

print("\n  CATEGORY I50 (heart failure) drilled to full codes:")
hf = circ10[circ10.category == "I50"]
for code, g in sorted(hf.groupby("icd_code"), key=lambda kv: -len(kv[1])):
    print(f"    {code:<8} {len(g):>7,} rows   {g.long_title.iloc[0][:72]}")

## 3. How many chapters, and how many unique codes in each?

This is the headline table: every chapter present in the data, how many **distinct codes** it
contains, how many **admissions** carry at least one of them, and how many **patients**.

Note that admissions counts sum to far more than the number of admissions — an admission
appears in every chapter it touches, which is the whole point of section 5.

In [ ]:
per_chapter = (dx.groupby("chapter")
                 .agg(distinct_codes=("icd_code", "nunique"),
                      categories=("category", "nunique"),
                      diagnosis_rows=("icd_code", "size"),
                      admissions=("hadm_id", "nunique"),
                      patients=("subject_id", "nunique"))
                 .sort_values("diagnosis_rows", ascending=False))

per_chapter["pct_of_admissions"] = (
    per_chapter.admissions / dx.hadm_id.nunique() * 100).round(1)

print(f"chapters present : {len(per_chapter)}")
print(f"distinct codes   : {dx.icd_code.nunique():,}")
print(f"distinct (code, version) pairs : {dx.groupby(['icd_code','icd_version']).ngroups:,}")
print(f"distinct categories : {dx.category.nunique():,}\n")
per_chapter

In [ ]:
# Where the codes are, versus where the patients are. A chapter can hold a huge
# number of rarely-used codes, or a handful of codes that nearly everyone gets.
fig_data = per_chapter.sort_values("distinct_codes", ascending=True)
ax = fig_data[["distinct_codes"]].plot(
    kind="barh", figsize=(9, 8), legend=False, color="#0F2A4A")
ax.set_xlabel("distinct ICD codes in the chapter")
ax.set_ylabel("")
ax.set_title("Unique ICD codes per chapter")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

In [ ]:
# ICD-9 vs ICD-10. MIMIC-IV spans the 2015 US transition, so both are present
# and the same condition appears under two different codes depending on the year.
ver = (dx.groupby("icd_version")
         .agg(rows=("icd_code", "size"), distinct_codes=("icd_code", "nunique"),
              admissions=("hadm_id", "nunique")))
ver["pct_of_rows"] = (ver.rows / len(dx) * 100).round(1)
print(ver, "\n")

both = (dx.groupby("long_title").icd_version.nunique() > 1).sum()
print(f"diagnosis titles that appear under BOTH ICD-9 and ICD-10: {both:,}")
print("-> the same clinical concept, split across two codes purely by coding era")

## 4. The long tail

Most codes are rare. This matters directly for modelling: a code used by three admissions
cannot support a learned effect, which is why models built on this data almost always collapse
diagnoses into counts or comorbidity indices rather than using the codes as features.

In [ ]:
code_freq = dx.icd_code.value_counts()
n_adm = dx.hadm_id.nunique()

print(f"distinct codes                 : {len(code_freq):,}")
print(f"used on exactly ONE admission  : {(code_freq == 1).sum():,} "
      f"({(code_freq == 1).mean():.1%} of codes)")
print(f"used on 10 or fewer            : {(code_freq <= 10).sum():,} "
      f"({(code_freq <= 10).mean():.1%})")
print(f"used on 1% or more of admissions: {(code_freq >= n_adm * 0.01).sum():,}\n")

cum = code_freq.cumsum() / code_freq.sum()
for share in (0.5, 0.8, 0.9):
    k = int((cum <= share).sum()) + 1
    print(f"  the top {k:>5,} codes ({k/len(code_freq):>5.1%} of them) cover {share:.0%} of all diagnosis rows")

print("\nTOP 20 CODES OVERALL")
top = (dx.groupby(["icd_code", "icd_version", "long_title", "chapter"])
         .hadm_id.nunique().sort_values(ascending=False).head(20)
         .rename("admissions").reset_index())
top["pct_of_admissions"] = (top.admissions / n_adm * 100).round(1)
top

In [ ]:
# The top codes WITHIN each chapter -- what each broad category actually contains.
TOP_N = 5
for chapter in per_chapter.index[:10]:
    sub = dx[dx.chapter == chapter]
    counts = (sub.groupby(["icd_code", "long_title"]).hadm_id.nunique()
                 .sort_values(ascending=False).head(TOP_N))
    print(f"\n{chapter.upper()}  ({sub.icd_code.nunique():,} distinct codes)")
    for (c, t), n in counts.items():
        print(f"   {c:<9} {n:>8,} admissions   {t[:76]}")

## 5. One disease or many?

The central question. `seq_num = 1` marks exactly one principal diagnosis per admission;
everything else coded on that stay is a secondary — comorbidities, complications, and conditions
managed during the stay.

In [ ]:
per_adm = dx.groupby("hadm_id").size()

print("DIAGNOSES CODED PER ADMISSION")
print(f"  mean {per_adm.mean():.1f}   median {per_adm.median():.0f}   "
      f"min {per_adm.min()}   max {per_adm.max()}")
print("  percentiles:", {f"p{p}": int(np.percentile(per_adm, p))
                         for p in (5, 10, 25, 50, 75, 90, 95, 99)})
print()
print(f"  exactly ONE diagnosis : {(per_adm == 1).sum():>8,}  ({(per_adm == 1).mean():.2%})")
for k in (2, 5, 10, 15, 20, 30):
    print(f"  {k:>2} or more           : {(per_adm >= k).sum():>8,}  ({(per_adm >= k).mean():.1%})")

ax = per_adm.clip(upper=40).plot(kind="hist", bins=40, figsize=(9, 4), color="#0F2A4A")
ax.set_xlabel("diagnoses coded on one admission (clipped at 40)")
ax.set_title("Almost no admission has a single diagnosis")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

In [ ]:
def show_admission(hadm_id, header):
    rows = dx[dx.hadm_id == hadm_id].sort_values("seq_num")
    print("=" * 100)
    print(f"{header}   |   admission {hadm_id}   |   {len(rows)} diagnoses")
    print("=" * 100)
    for _, r in rows.iterrows():
        role = "PRINCIPAL" if r.seq_num == 1 else f"  seq {int(r.seq_num):>2}"
        print(f"  {role}  {r.icd_code:<9} ICD-{r.icd_version}  {r.long_title[:74]}")
        print(f"{'':>14}{'':<9}         -> {r.chapter}")
    print(f"\n  chapters spanned: {rows.chapter.nunique()}   "
          f"categories spanned: {rows.category.nunique()}\n")

single = per_adm[per_adm == 1]
if len(single):
    show_admission(single.index[0], "SINGLE-DIAGNOSIS ADMISSION -- the 1% case")

typical = per_adm[per_adm == int(per_adm.median())]
show_admission(typical.index[0], "TYPICAL ADMISSION -- the median case")

show_admission(per_adm.idxmax(), "MOST COMPLEX ADMISSION IN THE DATA")

## 6. How far does one admission spread across the hierarchy?

A patient is not "a circulatory patient". Their codes scatter across the tree, and the number of
distinct **chapters** on one admission is a blunt but useful measure of how many organ systems
are in play at once.

In [ ]:
spread = (dx.groupby("hadm_id")
            .agg(diagnoses=("icd_code", "size"),
                 chapters=("chapter", "nunique"),
                 categories=("category", "nunique")))

print("DISTINCT CHAPTERS PER ADMISSION")
c = spread.chapters.value_counts().sort_index()
for k, n in c.items():
    print(f"  {k:>2} chapter(s) : {n:>8,}  ({n/len(spread):>6.1%})")
print(f"\n  2 or more   : {(spread.chapters >= 2).sum():>8,}  ({(spread.chapters >= 2).mean():.1%})")
print(f"  5 or more   : {(spread.chapters >= 5).sum():>8,}  ({(spread.chapters >= 5).mean():.1%})")
print(f"\n  mean chapters per admission   : {spread.chapters.mean():.1f}")
print(f"  mean categories per admission : {spread.categories.mean():.1f}")

In [ ]:
# Which chapters travel together. Co-occurrence is counted per ADMISSION: for
# each pair of chapters, how many admissions carry at least one code from both.
top_chapters = list(per_chapter.index[:12])
pairs = Counter()
for _, grp in dx[dx.chapter.isin(top_chapters)].groupby("hadm_id"):
    present = sorted(set(grp.chapter))
    for i in range(len(present)):
        for j in range(i + 1, len(present)):
            pairs[(present[i], present[j])] += 1

matrix = pd.DataFrame(0, index=top_chapters, columns=top_chapters, dtype=int)
for (a, b), n in pairs.items():
    matrix.loc[a, b] = n
    matrix.loc[b, a] = n

print("MOST COMMON CHAPTER PAIRS  (admissions carrying both)")
for (a, b), n in pairs.most_common(15):
    print(f"  {n:>8,}   {a}  +  {b}")

short = [c[:26] for c in matrix.columns]
styled = matrix.copy()
styled.index, styled.columns = short, short
styled

In [ ]:
# The same question one level down: which specific CODES co-occur. This is the
# comorbidity pattern a care programme would actually design around.
CAP = 200_000   # keep the pair explosion bounded on the full dataset
sample_adm = dx.hadm_id.drop_duplicates()
if len(sample_adm) > CAP:
    sample_adm = sample_adm.sample(CAP, random_state=42)
sub = dx[dx.hadm_id.isin(set(sample_adm))]

title_of = dict(zip(sub.icd_code, sub.long_title))
code_pairs = Counter()
for _, grp in sub.groupby("hadm_id"):
    codes = sorted(set(grp.icd_code))
    if len(codes) > 25:      # skip the extreme tail, it dominates the count
        continue
    for i in range(len(codes)):
        for j in range(i + 1, len(codes)):
            code_pairs[(codes[i], codes[j])] += 1

print(f"TOP 20 CO-OCCURRING CODE PAIRS  (from {sub.hadm_id.nunique():,} admissions)\n")
for (a, b), n in code_pairs.most_common(20):
    print(f"  {n:>7,}   {a:<8} {title_of[a][:42]:<44} + {b:<8} {title_of[b][:42]}")

## 7. Comorbidity burden per patient

Codes belong to admissions, so a patient with several stays accumulates a wider set than any
single admission shows.

In [ ]:
per_patient = dx.groupby("subject_id").agg(
    admissions=("hadm_id", "nunique"),
    diagnosis_rows=("icd_code", "size"),
    distinct_codes=("icd_code", "nunique"),
    distinct_chapters=("chapter", "nunique"))

print("PER PATIENT, ACROSS ALL THEIR ADMISSIONS")
print(per_patient.describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_string())
print()
print(f"  patients with 1 admission only : {(per_patient.admissions == 1).mean():.1%}")
print(f"  patients touching 5+ chapters  : {(per_patient.distinct_chapters >= 5).mean():.1%}")
print(f"  most codes carried by one patient : {per_patient.distinct_codes.max():,}")

## 8. Export

Everything above, written to `/kaggle/working` as CSVs. Download them from the notebook's
**Output** tab, or commit the notebook and pull the files from the version.

In [ ]:
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

per_chapter.to_csv(f"{OUT}/icd_chapters_summary.csv")

full_index = (dx.groupby(["chapter", "category", "icd_code", "icd_version", "long_title"])
                .agg(admissions=("hadm_id", "nunique"), patients=("subject_id", "nunique"))
                .reset_index()
                .sort_values(["chapter", "admissions"], ascending=[True, False]))
full_index.to_csv(f"{OUT}/icd_full_index.csv", index=False)

principal_index = (dx[dx.seq_num == 1]
                   .groupby(["chapter", "icd_code", "icd_version", "long_title"])
                   .hadm_id.nunique().rename("admissions").reset_index()
                   .sort_values("admissions", ascending=False))
principal_index.to_csv(f"{OUT}/icd_principal_index.csv", index=False)

spread.to_csv(f"{OUT}/admission_diagnosis_spread.csv")

print("written to", OUT)
for f in ("icd_chapters_summary.csv", "icd_full_index.csv",
          "icd_principal_index.csv", "admission_diagnosis_spread.csv"):
    print(f"  {f:<34} {os.path.getsize(f'{OUT}/{f}'):>12,} bytes")
print(f"\nfull index rows: {len(full_index):,}   principal-only rows: {len(principal_index):,}")

## 9. The numbers to bring back

Paste this block back into the Preventra project — it is the summary the documentation in
`docs/mimic/mimic_disease_landscape.md` is waiting on, measured on the full dataset rather than on the
2,000-patient loaded cohort.

In [ ]:
per_adm = dx.groupby("hadm_id").size()
lines = [
    "MIMIC-IV ICD SUMMARY",
    f"  source file            : {os.path.basename(dx_path)}  ({scale})",
    f"  diagnosis rows         : {len(dx):,}",
    f"  admissions             : {dx.hadm_id.nunique():,}",
    f"  patients               : {dx.subject_id.nunique():,}",
    "",
    f"  distinct ICD codes     : {dx.icd_code.nunique():,}",
    f"  distinct categories    : {dx.category.nunique():,}",
    f"  chapters present       : {dx.chapter.nunique()}",
    f"  ICD-9 / ICD-10 rows    : {(dx.icd_version == 9).sum():,} / {(dx.icd_version == 10).sum():,}",
    "",
    f"  diagnoses per admission: mean {per_adm.mean():.1f}, median {per_adm.median():.0f}, max {per_adm.max()}",
    f"  single-diagnosis stays : {(per_adm == 1).mean():.2%}",
    f"  10+ diagnoses          : {(per_adm >= 10).mean():.1%}",
    f"  chapters per admission : mean {spread.chapters.mean():.1f}, "
    f"{(spread.chapters >= 2).mean():.1%} span 2 or more",
    "",
    "  largest chapters by distinct codes:",
]
for chapter, row in per_chapter.sort_values("distinct_codes", ascending=False).head(8).iterrows():
    lines.append(f"    {chapter:<46} {int(row.distinct_codes):>6,} codes  "
                 f"{int(row.admissions):>9,} admissions")

report = "\n".join(lines)
print(report)
open(f"{OUT}/mimic_icd_summary.txt", "w").write(report + "\n")